112062308 蔡佳倩

Code explanation

Part 1: Import needed functions

In [ ]:
import re
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from textblob import TextBlob
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import warnings
warnings.filterwarnings("ignore")

nltk.download('stopwords')
stop = set(stopwords.words('english'))

Part 2 : Define the preprocessor

In [ ]:
# --- Preprocessing ---
def preprocessor(text):
    soup = BeautifulSoup(text, 'html.parser')
    
    time_tag = soup.find('time')
    post_datetime = None
    if time_tag and time_tag.has_attr('datetime'):
        post_datetime = time_tag['datetime']

    
    # channel
    article_tag = soup.find('article')
    channel = None
    if article_tag and article_tag.has_attr('data-channel'):
        channel = article_tag['data-channel']

    
    # category
    category_tags = soup.select('footer.article-topics a')
    categories = [tag.get_text(strip=True).lower() for tag in category_tags]

    text = soup.get_text()
    
    r = r'(?::|;|=|X)(?:-)?(?:\)|\(|D|P)'
    emoticons = re.findall(r, text)
    text = re.sub(r, '', text)
    text = re.sub('[\W]+', ' ', text.lower()) + ' ' + ' '.join(emoticons).replace('-', '')
    
    return text, post_datetime, channel, categories

1. First, I used BeautifulSoup to parse the input HTML text so I can easily extract specific elements of the html code from ‘soup’ the variance.
2. It searches for a <time> tag and retrieves the value of ‘datetime’ attribute. The publication time of the news would be saved in ‘post_datetime’.
3. It searches for a <article> tag and reads its ‘data-channel’ attribute. ‘channel’ identifies the news source.
4. It gathers all topic tags located within the article footer. Each category names are cleaned, turned as low case, and stored in a list.
5. Next is the text cleaning and emoticon handling process: remove hyphen from emoticons and non-alphanumeric characters from the text.
6. Last, return the cleaned text, post_datetime, channel, and categories for further use.

Part 3 : Define tokenizer_stem_nostop and extract_sentiment_features

In [ ]:
def tokenizer_stem_nostop(text):
    porter = PorterStemmer()
    return [porter.stem(w) for w in re.split('\s+', text.strip()) if w not in stop and re.match('[a-zA-Z]+', w)]


from textblob import TextBlob
def extract_sentiment_features(text_list):
    polarity = []
    subjectivity = []
    for text in text_list:
        blob = TextBlob(text)
        polarity.append(blob.sentiment.polarity)       # [-1, 1]，負面到正面
        subjectivity.append(blob.sentiment.subjectivity)  # [0, 1]，客觀到主觀
    return pd.DataFrame({
        'sentiment_polarity': polarity,
        'sentiment_subjectivity': subjectivity
    }).fillna(0)

tokenizer_stem_nostop
1. Initialize the Stemmer to perform stemming, which makes words transform to their base forms. Then split the text into tokens based on whitespaces. 
2. It also filters out words that are in the predefined stop word list and non-alphabetical strings. Then, return the result of each stemmed token.
 
extract_sentiment_features
1. Initialize two containers to store the sentiments results. Each word from the ‘text_list’ input would create a ‘TextBlob’ object. Then, it extracts the sentiment scores from negative to positive and from subjective to objective.
2. Results would be combined into a pandas DataFrame for structured analysis and any missing values would be filled with zero.

Part 4 : Converts a list of datetime strings into multiple numeric features

In [ ]:
def extract_time_features(datetime_list):
    # 加上 utc=True 來處理 tz-aware datetime
    dt_series = pd.Series(pd.to_datetime(datetime_list, errors='coerce', utc=True))
    
    df = pd.DataFrame()
    df['day_of_week'] = dt_series.dt.dayofweek
    df['month'] = dt_series.dt.month
    df['day'] = dt_series.dt.day
    df['hour'] = dt_series.dt.hour
    df['is_weekend'] = dt_series.dt.dayofweek.isin([5,6]).astype(int)   

    # 新增熱門時段特徵（18:00–22:00）
    df['is_peak_hour'] = df['hour'].between(18, 22).astype(int)
    # 新增工作日特徵
    df['is_workday'] = (~df['is_weekend']).astype(int)
    
    return df.fillna(0)

1. First, convert all datetime strings into pandas and make sure the time zone and context are a proper form of time.
2. Next, derive several time-related attributes (day of week, month, day, hour) from each datetime value. This provides a structured representation of each timestamp.
3. The weekend indicator uses a binary feature to mark whether the post was published on a weekend. The peak hour indicator also marks whether the post was published when people usually get off work. The workday indicator is just the opposite of the weekend indicator. I think these three indicators may influence the engagement and popularity of news, so I added it in the features.
4. At last, the function returns a pandas DataFrame containing all extracted features with any missing values filled with zeros.

Part 5 : Define the classifier and the preprocess of the training data

In [ ]:
# use randomforest
from sklearn.ensemble import RandomForestClassifier

# 初始化 RandomForest 模型
clf = RandomForestClassifier(
    n_estimators=100,       # 樹的數量，可調整
    max_depth=None,         # 樹的最大深度，可調整
    random_state=42,        # 為了結果可重現
    n_jobs=-1               # 使用所有 CPU 核心加速訓練
)

# 讀取所有訓練資料（不再用 chunk）
df = pd.read_csv('/kaggle/input/2025-data-lab-cup-1-predicting-news-popularity/train.csv')

# 前處理：取得文字與時間
processed = [preprocessor(t) for t in df['Page content']]
X_text = [p[0] for p in processed]
X_time_raw = [p[1] for p in processed] 
X_channel_raw = [p[2] for p in processed]
X_categories_raw = [p[3] for p in processed]

y = df['Popularity']

1. After different tries, I chose to use a Random Forest Classifier as the main model. First, initializing it with the parameters above. Then, read in the training dataset.
2. Before training, multiple feature types are extracted from the dataset by the preprocessor defined above. Get the return values and define y as the known popularity of each article.

Part 6 : Define the text representation, train the model and evaluate it

In [ ]:
#BoW處理文字
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(
    tokenizer=tokenizer_stem_nostop,
    max_features=100000,
    ngram_range=(1,2)
)
X_vec_bow = bow.fit_transform(X_text)

# 將 channel 做 One-Hot 編碼
from sklearn.preprocessing import OneHotEncoder

channel_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
#channel_encoder.fit(np.array(X_channel_raw).reshape(-1, 1))
X_channel = channel_encoder.fit_transform(np.array(X_channel_raw).reshape(-1, 1))

# 將 category 做 MultiLabel Binarization
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
X_categories = mlb.fit_transform(X_categories_raw)

# 訓練集 sentiment 特徵
X_sentiment = extract_sentiment_features(X_text)
                        
# 萃取時間特徵
X_time = extract_time_features(X_time_raw)

# 合併特徵
from scipy.sparse import hstack, csr_matrix
#X_text_combined = hstack([X_vec_bow, X_vec_tfidf])
X_combined = hstack([X_vec_bow, X_time.values, csr_matrix(X_channel), csr_matrix(X_categories), csr_matrix(X_sentiment.values)])

# 拆分訓練與驗證集
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# 訓練模型
clf.fit(X_train, y_train)

# 驗證
from sklearn.metrics import roc_auc_score
val_score = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
print(f'Validation AUC with BoW: {val_score:.4f}')

1. After different tries, I chose to use the Bag-of-Word model with ‘CountVectorizer’ : including the predefined tokenizer, both unigrams and bigrams, vocabulary limit.
2. Encode the channel information using one-hot encoder and convert each channel into a separate binary feature column.
3. Since each article can belong to multiple categories, ‘MultiLabelBinarizer’ is used to perform multi-hot encoding generating binary indicators for all category tags.
4. Then, use the predefined functions to extract their sentiment analysis and time features. At last, concatenate every extracted feature into one unified feature matrix.
5. The training dataset is split into 80% training sets and 20% validation sets to evaluate the model performance. Use the AUC metric to indicate whether the generation of the popularity is precise.

Part 7 : Run the testing dataset and output the prediction in a csv file.

In [ ]:
# 讀取測試資料
test_df = pd.read_csv('/kaggle/input/2025-data-lab-cup-1-predicting-news-popularity/test.csv')

# 對每篇文章做前處理，取得文字與時間
processed_test = [preprocessor(t) for t in test_df['Page content']]
X_test_text = [p[0] for p in processed_test]
X_test_time_raw = [p[1] for p in processed_test]
X_test_channel_raw = [p[2] for p in processed_test]
X_test_categories_raw = [p[3] for p in processed_test]

# 文字向量化
#X_test_tfidf = tfidf.transform(X_test_text)
X_test_bow = bow.transform(X_test_text)

# 時間特徵萃取
X_test_time = extract_time_features(X_test_time_raw)
X_test_channel = channel_encoder.transform(np.array(X_test_channel_raw).reshape(-1, 1))
X_test_categories = mlb.transform(X_test_categories_raw)
X_test_sentiment = extract_sentiment_features(X_test_text)

# 合併文字向量與時間特徵
#X_test_vec = hstack([X_test_bow, X_test_tfidf])
X_test_combined = hstack([X_test_bow, X_test_time.values, csr_matrix(X_test_channel), csr_matrix(X_test_categories), csr_matrix(X_test_sentiment.values)])

# 預測機率
y_pred = clf.predict_proba(X_test_combined)[:, 1]

# 建立提交檔案
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'Popularity': np.round(y_pred, 1)
})
submission.to_csv('1014_submission_bow_tfidf_rf_sentiment.csv', index=False)

Read the testing dataset and do every preprocess just as the training dataset just did. Save the probability prediction in a panda DataFrame first with ‘Id’ and ‘Popularity’ then turn the results into a csv output file.

Conclusion: 

1. I tried different models for classifiers and text representation then chose the best performance combination. I observed that some combinations seem to be facing overfitting with a better performance locally but a worse performance at the testing dataset. Below are the other models I tried during the trial session.

In [ ]:
# --- 使用 XGBoost ---
from xgboost import XGBClassifier
clf = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='auc', use_label_encoder=False, n_jobs=-1, random_state=42)


# use lightgbm
import lightgbm as lgb
clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, 
                         min_data_in_leaf=20, random_state=42, n_jobs=-1)

In [ ]:
# 向量化文字 (hashvectorize)
from sklearn.feature_extraction.text import HashingVectorizer
hashvec = HashingVectorizer(
    n_features=2**18,
    preprocessor=None,
    tokenizer=tokenizer_stem_nostop,
    ngram_range=(1,1)
)
X_vec = hashvec.transform(X_text)

# 向量化文字 (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    tokenizer=tokenizer_stem_nostop,
    max_features=100000,  # 限制詞彙數量
    ngram_range=(1,2),    # 使用 unigram + bigram
    sublinear_tf=True
)
X_vec_tfidf = tfidf.fit_transform(X_text)

2. I faced a pretty big challenge when doing the feature engineering. First, I don’t know what features may be effective to the model training. Sometimes adding too much feature to the training process makes it more complex and doesn’t even lead to a better result. Next, I’m not sure which model suits this kind of training session and this big amount of data, since I’m not familiar with any of these models. Only by trial and error, also Internet sources, can give me the answers. But too bad there are limited submissions a day. Last, after having a basic structure, thinking what other features can be extracted to help enhance the performance is also a big task in this competition. To sum up, I think that maintaining patience and keeping my mind clear about what I’m doing are the most difficult parts, but this is also a learning opportunity for me, a deep learning rookie.